## Decision Trees from Scratch

This notebook will demonstrate how to build a decision tree algorithm from the ground up, using a simple student performance dataset.

### 1. Sample Student Data

Let's create a simple dataset representing student performance, including marks, assignment marks, study hours, and a 'Pass' or 'Fail' outcome. This will be our target variable.

In [1]:
import pandas as pd
import numpy as np

# Sample student data
data = {
    'Marks': [85, 90, 70, 60, 75, 95, 50, 80, 65, 78],
    'Assignment_Marks': [90, 85, 75, 65, 80, 92, 55, 88, 70, 82],
    'Study_Hours': [8, 10, 6, 4, 7, 11, 3, 9, 5, 7],
    'Outcome': ['Pass', 'Pass', 'Pass', 'Fail', 'Pass', 'Pass', 'Fail', 'Pass', 'Fail', 'Pass']
}

df = pd.DataFrame(data)
display(df.head())


,Marks,Assignment_Marks,Study_Hours,Outcome
0,85,90,8,Pass
1,90,85,10,Pass
2,70,75,6,Pass
3,60,65,4,Fail
4,75,80,7,Pass


### 2. Decision Tree Concepts

To build a decision tree, we need to understand a few core concepts:

*   **Entropy**: A measure of impurity or disorder in a set of examples. High entropy means a mixed set of outcomes, while low entropy (or zero) means all outcomes are the same.
*   **Information Gain**: The reduction in entropy achieved by splitting a dataset on a particular feature. We want to choose the feature that provides the highest information gain.
*   **Splitting Criteria**: How we decide where to split the data (e.g., for numerical features, finding an optimal threshold; for categorical, splitting by categories).


#### 2.1 Entropy Calculation

Let's implement a function to calculate entropy.

In [2]:
def calculate_entropy(data_column):
    """Calculates the entropy of a given column (target variable)."""
    # Get value counts of unique outcomes
    counts = data_column.value_counts()
    # Calculate probabilities
    probabilities = counts / len(data_column)
    # Calculate entropy using the formula: -sum(p * log2(p))
    entropy = -np.sum(probabilities * np.log2(probabilities))
    return entropy

# Test with our 'Outcome' column
initial_entropy = calculate_entropy(df['Outcome'])
print(f"Initial Entropy of 'Outcome': {initial_entropy:.4f}")


Initial Entropy of 'Outcome': 0.8813


#### 2.2 Information Gain Calculation

Now, let's implement information gain. This function will calculate how much the entropy decreases if we split the data based on a particular feature.

In [3]:
def calculate_information_gain(data, feature_name, target_name):
    """Calculates the information gain for a given feature."""
    # Calculate initial entropy of the target variable
    initial_entropy = calculate_entropy(data[target_name])

    # Calculate entropy after splitting by the feature
    weighted_avg_entropy = 0
    for value in data[feature_name].unique():
        subset = data[data[feature_name] == value]
        # Calculate entropy of the subset
        subset_entropy = calculate_entropy(subset[target_name])
        # Weight by the proportion of data in the subset
        weighted_avg_entropy += (len(subset) / len(data)) * subset_entropy

    # Information Gain = Initial Entropy - Weighted Average Entropy
    information_gain = initial_entropy - weighted_avg_entropy
    return information_gain

# For numerical features like 'Marks', we need to discretize or find a split point.
# For simplicity, let's convert 'Marks' into categories first for this example.
# In a real scenario, you'd find optimal split points for numerical features.

df_categorical = df.copy()
df_categorical['Marks_Category'] = pd.cut(df_categorical['Marks'], bins=[0, 60, 75, 100], labels=['Low', 'Medium', 'High'])
df_categorical['Assignment_Marks_Category'] = pd.cut(df_categorical['Assignment_Marks'], bins=[0, 65, 80, 100], labels=['Low', 'Medium', 'High'])
df_categorical['Study_Hours_Category'] = pd.cut(df_categorical['Study_Hours'], bins=[0, 5, 8, 12], labels=['Low', 'Medium', 'High'])

print("Information Gain for 'Marks_Category':", calculate_information_gain(df_categorical, 'Marks_Category', 'Outcome'))
print("Information Gain for 'Assignment_Marks_Category':", calculate_information_gain(df_categorical, 'Assignment_Marks_Category', 'Outcome'))
print("Information Gain for 'Study_Hours_Category':", calculate_information_gain(df_categorical, 'Study_Hours_Category', 'Outcome'))


Information Gain for 'Marks_Category': 0.6058021490143459
Information Gain for 'Assignment_Marks_Category': 0.6058021490143459
Information Gain for 'Study_Hours_Category': 0.8812908992306927


### 3. Building the Decision Tree

Now, let's put it all together to build the decision tree. We'll represent the tree as a dictionary.

In [4]:
class Node:
    def __init__(self, feature=None, threshold=None, value=None, left=None, right=None):
        self.feature = feature        # Feature to split on (if not a leaf node)
        self.threshold = threshold    # Threshold for numerical splits
        self.value = value            # Class label if it's a leaf node
        self.left = left              # Left child node
        self.right = right            # Right child node

    def is_leaf(self):
        return self.value is not None

def find_best_split(data, features, target_name):
    """Finds the best feature and split point to divide the data."""
    best_feature = None
    best_threshold = None
    max_info_gain = -1
    initial_entropy = calculate_entropy(data[target_name])

    for feature in features:
        if data[feature].dtype in ['int64', 'float64']:
            # For numerical features, iterate through possible thresholds
            unique_values = sorted(data[feature].unique())
            for i in range(len(unique_values) - 1):
                threshold = (unique_values[i] + unique_values[i+1]) / 2
                left_subset = data[data[feature] <= threshold]
                right_subset = data[data[feature] > threshold]

                if len(left_subset) == 0 or len(right_subset) == 0:
                    continue

                # Calculate weighted entropy for this split
                left_entropy = calculate_entropy(left_subset[target_name])
                right_entropy = calculate_entropy(right_subset[target_name])

                weighted_entropy = (len(left_subset) / len(data)) * left_entropy + \
                                   (len(right_subset) / len(data)) * right_entropy

                info_gain = initial_entropy - weighted_entropy

                if info_gain > max_info_gain:
                    max_info_gain = info_gain
                    best_feature = feature
                    best_threshold = threshold
        else:
            # For categorical features, we treat each category as a split (simplified)
            # This can be more complex, but for this example, we'll use Information Gain as a primary metric
            gain = calculate_information_gain(data, feature, target_name)
            if gain > max_info_gain:
                max_info_gain = gain
                best_feature = feature
                best_threshold = None # No specific threshold for categorical splits in this simplified version

    return best_feature, best_threshold, max_info_gain

def build_tree(data, features, target_name, max_depth=None, current_depth=0):
    """Recursively builds the decision tree."""
    # Base cases:
    # 1. If all outcomes are the same, return a leaf node with that outcome
    if len(data[target_name].unique()) == 1:
        return Node(value=data[target_name].iloc[0])

    # 2. If no more features to split on, or max_depth reached, return a leaf node with the most common outcome
    if len(features) == 0 or (max_depth is not None and current_depth >= max_depth):
        return Node(value=data[target_name].mode()[0])

    # Find the best feature to split on
    best_feature, best_threshold, max_info_gain = find_best_split(data, features, target_name)

    # If no gain or no best feature found, return a leaf node with the most common outcome
    if best_feature is None or max_info_gain <= 0:
        return Node(value=data[target_name].mode()[0])

    # Create an internal node
    node = Node(feature=best_feature, threshold=best_threshold)

    if best_threshold is not None: # Numerical split
        left_data = data[data[best_feature] <= best_threshold]
        right_data = data[data[best_feature] > best_threshold]
    else: # Categorical split (simplified: for each category value)
        # For categorical, we'll create a dictionary of children for each category value
        # For this simple example, let's treat it as a binary split if we had a specific binary categorical feature
        # For multi-valued categorical, we'd need a more complex branching structure.
        # Let's just pick one value for now, or revert to mode if no clear binary split.
        # To simplify, we'll assume binary splitting for all features in this basic tree. For multi-class, more advanced splitting is needed.
        # Let's adjust find_best_split to handle categorical info_gain better.
        # For now, let's assume if best_threshold is None, it implies a categorical split, and we'd need to refine this.
        # For this simple example, we will treat all features as if they could be numerically split.
        # If we had a truly categorical feature like 'Gender', it would be handled differently.
        pass # This part needs refinement for truly categorical features without ordering/thresholds.

    # Recursively build left and right subtrees
    # Remove the split feature from the list of features for the next level if it's already used for a binary split
    remaining_features = [f for f in features if f != best_feature]

    # If a split is determined (e.g., numerical threshold)
    if best_threshold is not None:
        node.left = build_tree(left_data, remaining_features, target_name, max_depth, current_depth + 1)
        node.right = build_tree(right_data, remaining_features, target_name, max_depth, current_depth + 1)
    else:
        # For simplified categorical features, we can just take the most common for now
        # This needs a better strategy for true categorical features or multiple branches.
        node.value = data[target_name].mode()[0] # Fallback if categorical split logic isn't fully developed

    return node


def predict_single(tree, row):
    """Predicts the outcome for a single data point."""
    if tree.is_leaf():
        return tree.value

    feature_value = row[tree.feature]
    if tree.threshold is not None: # Numerical split
        if feature_value <= tree.threshold:
            return predict_single(tree.left, row)
        else:
            return predict_single(tree.right, row)
    else:
        # This part needs to be more sophisticated for true categorical splits
        # For now, it will likely hit a leaf node or a fallback.
        return tree.value # Should not happen if tree is built correctly with splits


def predict(tree, test_data):
    """Predicts outcomes for a test dataset."""
    predictions = [predict_single(tree, row) for index, row in test_data.iterrows()]
    return predictions


# Prepare data for building the tree
X = df[['Marks', 'Assignment_Marks', 'Study_Hours']]
y = df['Outcome']

# Split data into training and testing (simple split for demonstration)
# In a real scenario, use train_test_split from sklearn

train_df = df.iloc[:8]
test_df = df.iloc[8:]

tree = build_tree(train_df, ['Marks', 'Assignment_Marks', 'Study_Hours'], 'Outcome', max_depth=3)

print("\nDecision Tree (simplified structure for printing):")
def print_tree(node, indent=""):
    if node.is_leaf():
        print(f"{indent}Predict: {node.value}")
    else:
        print(f"{indent}{node.feature} <= {node.threshold}?")
        print(f"{indent}  ├── True:")
        print_tree(node.left, indent + "  │  ")
        print(f"{indent}  └── False:")
        print_tree(node.right, indent + "  │  ")

print_tree(tree)



Decision Tree (simplified structure for printing):
Marks <= 65.0?
  ├── True:
  │  Predict: Fail
  └── False:
  │  Predict: Pass


### 4. Making Predictions

Let's use our trained decision tree to make predictions on new (or unseen) student data.

In [5]:
# Make predictions on the test set
predictions = predict(tree, test_df[['Marks', 'Assignment_Marks', 'Study_Hours']])

# Evaluate accuracy (simple comparison)
actual_outcomes = test_df['Outcome'].tolist()

correct_predictions = 0
for i in range(len(predictions)):
    if predictions[i] == actual_outcomes[i]:
        correct_predictions += 1

accuracy = correct_predictions / len(predictions)

print(f"\nTest Data:\n{test_df}")
print(f"\nPredictions: {predictions}")
print(f"Actual Outcomes: {actual_outcomes}")
print(f"Accuracy: {accuracy * 100:.2f}%")



Test Data:
   Marks  Assignment_Marks  Study_Hours Outcome
8     65                70            5    Fail
9     78                82            7    Pass

Predictions: ['Fail', 'Pass']
Actual Outcomes: ['Fail', 'Pass']
Accuracy: 100.00%


### Conclusion

This implementation demonstrates the fundamental concepts of building a decision tree from scratch, including entropy, information gain, and recursive splitting. While simplified, it provides a solid foundation for understanding how these algorithms work. For production-level use, libraries like `scikit-learn` provide highly optimized and robust decision tree implementations.